### This notebook answers the following questions

Which dimensionality reduction method is best generalized?

Which vectorizer method is best generalized?

Which clustering algorithm is best generalized?

Which dataset showed the best overall results?

In [1]:
from os import makedirs
from datetime import date

import pandas as pd

today = date.today().strftime("%Y-%m-%d")
DEFAULT_SAVE_PATH = f"../results/{today}/"

df = pd.read_csv(f"{DEFAULT_SAVE_PATH}/best_configuration_per_dataset.csv")

In [2]:
df.head(8)

,dataset,vectorizer,dim_reduction,n_comp,K,alg,CHI,DBI,SIL,CV_COHERENCE,NPMI_COHERENCE,top_keywords
0,CS,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,3,KMeans,310.290985,1.024360,0.585291,0.476117,-0.272126,"{0: ['adversarial attack', 'adversarial exampl..."
1,PHYS,sentence-transformers/all-MiniLM-L6-v2,UMAP,15,10,KMeans,129.480255,1.071441,0.511686,0.459188,-0.267792,"{0: ['quantum simulator', 'quantum simulation'..."
2,MATH,sentence-transformers/all-MiniLM-L6-v2,UMAP,5,4,FCM,406.232391,0.917180,0.576935,0.739944,-0.212912,"{0: ['fluid dynamic cfd', 'reduced order model..."
3,EESS,sentence-transformers/all-MiniLM-L6-v2,UMAP,15,9,Agglomerative,145.653366,0.937027,0.486276,0.471749,-0.217433,"{0: ['multi scale', '3d medical image', 'encod..."
4,STAT,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,6,Agglomerative,105.214607,0.987736,0.472974,0.384808,-0.289326,"{0: ['causal effect', 'causal inference', 'sam..."
5,Q_BIO,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,10,Agglomerative,65.970016,0.982325,0.471244,0.437038,-0.217307,"{0: ['electrical stimulation', 'real time', 'b..."
6,Q_FIN,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,4,Agglomerative,528.420959,0.808184,0.590241,0.588879,-0.188780,"{0: ['deep reinforcement learn', 'financial ti..."
7,ECON,sentence-transformers/all-MiniLM-L6-v2,UMAP,10,11,KMeans,123.945793,0.869383,0.592780,0.387135,-0.285966,"{0: ['systemic risk', 'impulse response', 'mon..."


#### Which dataset showed the best overall results in the analyses? 

Q_FIN has the highest CHI (528.42), the lowest DBI (0.808), second highest Silhouette (0.590), second highest CV Coherence (0.589), best NPMI Coherence (−0.189)

The second best is MATH


In [3]:
print("\nVectorizer frequency:")
print(df["vectorizer"].value_counts())

print("\nDimensionality reduction frequency:")
print(df["dim_reduction"].value_counts())

print("\nClustering algorithm frequency:")
print(df["alg"].value_counts())

print("\nn_components frequency:")
print(df["n_comp"].value_counts())


Vectorizer frequency:
vectorizer
sentence-transformers/all-MiniLM-L6-v2    8
Name: count, dtype: int64

Dimensionality reduction frequency:
dim_reduction
UMAP    8
Name: count, dtype: int64

Clustering algorithm frequency:
alg
Agglomerative    4
KMeans           3
FCM              1
Name: count, dtype: int64

n_components frequency:
n_comp
10    5
15    2
5     1
Name: count, dtype: int64


#### Which dimensionality reduction method is best generalized?
**UMAP** outperfomed SVD

In [5]:
dimred_avg = (
    df.groupby("dim_reduction")[["SIL","CHI","DBI","CV_COHERENCE","NPMI_COHERENCE"]]
    .mean()
    .reset_index()
)

print("\nDimensionality reduction average metrics:")
print(dimred_avg)


Dimensionality reduction average metrics:
  dim_reduction       SIL         CHI       DBI  CV_COHERENCE  NPMI_COHERENCE
0          UMAP  0.535928  226.901047  0.949704      0.493107       -0.243955


#### Which vectorizer method is best generalized?
**sentence-transformers/all-MiniLM-L6-v2** outperfomed TF-IDF and SciBERT

In [4]:
vectorizer_avg = (
    df.groupby("vectorizer")[["SIL","CHI","DBI","CV_COHERENCE","NPMI_COHERENCE"]]
    .mean()
    .reset_index()
)

print("\nVectorizer average metrics:")
print(vectorizer_avg)


Vectorizer average metrics:
                               vectorizer       SIL         CHI       DBI  \
0  sentence-transformers/all-MiniLM-L6-v2  0.535928  226.901047  0.949704   

   CV_COHERENCE  NPMI_COHERENCE  
0      0.493107       -0.243955  


### Comparison between Agglomerative and KMeans

Since the frequency difference between Agglomerative and K-Means is 1 we compare the average metrics.

#### Which clustering algorithm is best generalized?

Agglomerative outperforms KMeans in CHI, DBI, CV_Coherence and NPMI_Coherence.

In [6]:
alg_avg = (
    df.groupby("alg")[["SIL","CHI","DBI","CV_COHERENCE","NPMI_COHERENCE"]]
    .mean()
    .reset_index()
)

print("\nAlgorithm average metrics:")
print(alg_avg)


Algorithm average metrics:
             alg       SIL         CHI       DBI  CV_COHERENCE  NPMI_COHERENCE
0  Agglomerative  0.505184  211.314737  0.928818      0.470618       -0.228212
1            FCM  0.576935  406.232391  0.917180      0.739944       -0.212912
2         KMeans  0.563252  187.905678  0.988395      0.440813       -0.275295


#### **n_component=10** outperfomed 5 and 15 because of the frequency

In [7]:
ncomp_avg = (
    df.groupby("n_comp")[["SIL","CHI","DBI","CV_COHERENCE","NPMI_COHERENCE"]]
    .mean()
    .reset_index()
)

print("\nn_components average metrics:")
print(ncomp_avg)


n_components average metrics:
   n_comp       SIL         CHI       DBI  CV_COHERENCE  NPMI_COHERENCE
0       5  0.576935  406.232391  0.917180      0.739944       -0.212912
1      10  0.542506  226.768472  0.934398      0.454795       -0.250701
2      15  0.498981  137.566811  1.004234      0.465468       -0.242612


### Choosing the pipeline for the app

We identify the most consistent pattern by considering:

- **1-Frequency between datasets**
- **2-Metrics averages**

Most generalized configuration:
**UMAP, n_component=10, sentence-transformers/all-MiniLM-L6-v2, Agglomerative Cluster**